# Central Orchestrator for the Event-Planning Multi-Agent System

This notebook adds an **orchestrator** on top of the four single-purpose agents from
`Task.ipynb` (`VenueAgent`, `CateringLogisticsAgent`, `AVVendorCoordinationAgent`,
`AgendaAssemblyAgent`). The orchestrator:

1. Interprets one user-facing `EventRequest`.
2. Decides which agents to call and in what order (a DAG, not a fixed pipeline).
3. Routes each agent's output into the next agent's input (e.g. the venue agent's choice feeds
   into the AV agent, and venue + catering + AV all feed into the agenda agent).
4. Runs at least one branch **in parallel** (venue/AV sourcing happens concurrently with
   catering sourcing).
5. Makes at least one **conditional routing decision** (skip `VenueAgent` entirely if the client
   already has a booked space).
6. Produces an end-to-end **trace log** of every agent handoff.

**Deliverables**
- An orchestrator (`EventPlanningOrchestrator`) that routes between all four agents.
- An ASCII (+ Mermaid) agent-flow diagram of the call DAG.
- An end-to-end trace log for a sample event request, plus a second run that exercises the
  conditional "existing venue" branch.

As in `Task.ipynb` / `Task2.ipynb`, every agent depends only on the duck-typed `LLMClient`
protocol, so the whole notebook runs deterministically with no network access or API
credentials, via a scripted `FakeLLMClient`.

In [1]:
import json
import os
import sys
import time
from concurrent.futures import ThreadPoolExecutor
from enum import Enum
from typing import Any, Dict, List, Literal, Optional, Protocol, Tuple, Type

from pydantic import BaseModel, Field

# Make the repo-root `init_creds.py` importable - only needed if/when `AzureOpenAIClient` below
# is actually instantiated.
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)


class EventType(str, Enum):
    """Shared event taxonomy used as input by every agent (same as `Task.ipynb`)."""

    CONFERENCE = "conference"
    WEDDING = "wedding"
    CORPORATE_OFFSITE = "corporate_offsite"
    PRODUCT_LAUNCH = "product_launch"
    GALA = "gala"
    WORKSHOP = "workshop"
    OTHER = "other"


class LLMClient(Protocol):
    """Minimal interface every agent depends on (same contract as `Task.ipynb`)."""

    def generate(
        self,
        system_prompt: str,
        user_payload: Dict[str, Any],
        schema_name: str,
        json_schema: Dict[str, Any],
    ) -> Dict[str, Any]:
        """Return a dict that should validate against `json_schema`."""
        ...


class AzureOpenAIClient:
    """Production LLM backend: Azure OpenAI chat completions with structured JSON output.

    Credentials are resolved lazily (via the repo's `init_creds.py` helpers) only when an
    instance is actually constructed, so importing/using this module never requires network
    access or secrets.
    """

    def __init__(self, model: str = "gpt-4o-mini", api_version: str = "2025-04-01-preview"):
        from openai import AzureOpenAI
        import init_creds as creds

        self._client = AzureOpenAI(
            api_key=creds.get_api_key(),
            azure_endpoint=creds.get_endpoint(),
            api_version=api_version,
        )
        self._model = model

    def generate(self, system_prompt, user_payload, schema_name, json_schema):
        response = self._client.chat.completions.create(
            model=self._model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": json.dumps(user_payload)},
            ],
            response_format={
                "type": "json_schema",
                "json_schema": {"name": schema_name, "schema": json_schema, "strict": False},
            },
        )
        return json.loads(response.choices[0].message.content)


class FakeLLMClient:
    """Deterministic stand-in for a real LLM backend, keyed by `schema_name` (= each agent's
    `name`) so a single instance can back all four different agents in the orchestrator demo.

    `delay_seconds` simulates realistic network/inference latency - with it set, the parallel
    branch below becomes measurably faster than running every agent call sequentially.
    """

    def __init__(self, canned_responses: Dict[str, Dict[str, Any]], delay_seconds: float = 0.0):
        self.canned_responses = canned_responses
        self.delay_seconds = delay_seconds
        self.calls: List[Dict[str, Any]] = []

    def generate(self, system_prompt, user_payload, schema_name, json_schema):
        if self.delay_seconds:
            time.sleep(self.delay_seconds)
        self.calls.append({"schema_name": schema_name, "user_payload": user_payload})
        if schema_name not in self.canned_responses:
            raise KeyError(f"No canned response registered for '{schema_name}'")
        return self.canned_responses[schema_name]

## 1. The four specialist agents (reused from `Task.ipynb`)

Same `BaseAgent` scaffolding and the same four agent classes/schemas as `Task.ipynb` - repeated
here so this notebook is self-contained and runnable on its own.

In [2]:
class BaseAgent:
    """Shared scaffolding for single-purpose, structured-output agents."""

    name: str
    system_prompt: str
    input_model: Type[BaseModel]
    output_model: Type[BaseModel]

    def __init__(self, llm: LLMClient):
        self._llm = llm

    def run(self, request):
        if not isinstance(request, self.input_model):
            request = self.input_model.model_validate(request)
        payload = request.model_dump(mode="json")
        raw_output = self._llm.generate(
            system_prompt=self.system_prompt,
            user_payload=payload,
            schema_name=self.name,
            json_schema=self.output_model.model_json_schema(),
        )
        return self.output_model.model_validate(raw_output)


class VenueRequest(BaseModel):
    event_type: EventType
    headcount: int = Field(gt=0)
    budget_usd: float = Field(gt=0)
    location: str = Field(min_length=1)
    event_date: Optional[str] = None
    special_requirements: List[str] = Field(default_factory=list)


class VenueOption(BaseModel):
    name: str
    area: str
    capacity: int = Field(gt=0)
    estimated_cost_usd: float = Field(ge=0)
    amenities: List[str] = Field(default_factory=list)
    rationale: str


class VenueRecommendation(BaseModel):
    recommended_venues: List[VenueOption] = Field(min_length=1)
    assumptions: List[str] = Field(default_factory=list)
    notes: Optional[str] = None


class VenueAgent(BaseAgent):
    name = "venue_recommendation_agent"
    input_model = VenueRequest
    output_model = VenueRecommendation
    system_prompt = (
        "You are a single-purpose venue recommendation specialist for live events. Propose "
        "2-4 realistic candidate venues from headcount, budget, location, and event type. "
        "Respond ONLY with the structured JSON object defined by the provided schema."
    )


class CateringRequest(BaseModel):
    event_type: EventType
    headcount: int = Field(gt=0)
    budget_usd: float = Field(gt=0)
    meal_period: Literal["breakfast", "brunch", "lunch", "dinner", "reception"]
    dietary_restrictions: List[str] = Field(default_factory=list)
    service_style_preference: Optional[
        Literal["plated", "buffet", "family_style", "food_stations", "cocktail"]
    ] = None


class MenuItem(BaseModel):
    course: str
    description: str
    dietary_tags: List[str] = Field(default_factory=list)


class CateringPlan(BaseModel):
    service_style: str
    menu_items: List[MenuItem] = Field(min_length=1)
    staffing_notes: str
    estimated_cost_usd: float = Field(ge=0)
    cost_per_head_usd: float = Field(ge=0)
    assumptions: List[str] = Field(default_factory=list)


class CateringLogisticsAgent(BaseAgent):
    name = "catering_logistics_agent"
    input_model = CateringRequest
    output_model = CateringPlan
    system_prompt = (
        "You are a single-purpose catering and food-service logistics specialist. Propose ONE "
        "coherent catering plan covering every dietary restriction supplied. Respond ONLY with "
        "the structured JSON object defined by the provided schema."
    )

In [3]:
class AVRequest(BaseModel):
    event_type: EventType
    headcount: int = Field(gt=0)
    budget_usd: float = Field(gt=0)
    venue_type: Optional[str] = None
    special_requirements: List[str] = Field(default_factory=list)


class AVSetupItem(BaseModel):
    category: str
    description: str
    quantity: int = Field(ge=1)


class VendorSuggestion(BaseModel):
    vendor_role: str
    notes: str


class AVPlan(BaseModel):
    av_setup: List[AVSetupItem] = Field(min_length=1)
    recommended_vendors: List[VendorSuggestion] = Field(min_length=1)
    estimated_cost_usd: float = Field(ge=0)
    setup_lead_time_hours: float = Field(ge=0)
    assumptions: List[str] = Field(default_factory=list)


class AVVendorCoordinationAgent(BaseAgent):
    name = "av_vendor_coordination_agent"
    input_model = AVRequest
    output_model = AVPlan
    system_prompt = (
        "You are a single-purpose audiovisual and vendor-coordination specialist. Propose ONE "
        "AV setup plan and the vendors needed to deliver it, covering every special requirement. "
        "Respond ONLY with the structured JSON object defined by the provided schema."
    )


class AgendaRequest(BaseModel):
    event_name: str
    event_date: str
    start_time: str
    end_time: str
    venue: VenueRecommendation
    catering: CateringPlan
    av: AVPlan


class RunOfShowItem(BaseModel):
    start_time: str
    end_time: str
    activity: str
    owner: str
    location: Optional[str] = None
    notes: Optional[str] = None


class RunOfShow(BaseModel):
    event_name: str
    event_date: str
    selected_venue: str
    run_of_show: List[RunOfShowItem] = Field(min_length=1)
    overall_notes: List[str] = Field(default_factory=list)


class AgendaAssemblyAgent(BaseAgent):
    name = "agenda_assembly_agent"
    input_model = AgendaRequest
    output_model = RunOfShow
    system_prompt = (
        "You are a single-purpose agenda-assembly specialist. Choose exactly ONE venue from "
        "`venue.recommended_venues` and sequence catering + AV + main activities into a single "
        "chronological run-of-show. Respond ONLY with the structured JSON object defined by the "
        "provided schema."
    )

## 2. The orchestrator's input: `EventRequest`

One user-facing request. `existing_venue` is the hook for the **conditional routing decision**:
when it's set, the orchestrator skips `VenueAgent` entirely and builds a `VenueRecommendation`
directly from the client-supplied venue.

In [4]:
class ExistingVenue(BaseModel):
    """Supplied by the client when a venue is already booked - triggers the orchestrator's
    conditional routing to skip `VenueAgent` entirely."""

    name: str
    area: str
    capacity: int = Field(gt=0)
    amenities: List[str] = Field(default_factory=list)


class EventRequest(BaseModel):
    """The single user-facing request the orchestrator interprets and fans out to the four
    specialist agents."""

    event_name: str
    event_type: EventType
    headcount: int = Field(gt=0)
    budget_usd: float = Field(
        gt=0, description="Total budget; the orchestrator allocates a share of it to each agent."
    )
    location: str
    event_date: str = Field(description="ISO date (YYYY-MM-DD)")
    start_time: str = Field(description="24h HH:MM")
    end_time: str = Field(description="24h HH:MM")
    meal_period: Literal["breakfast", "brunch", "lunch", "dinner", "reception"]
    dietary_restrictions: List[str] = Field(default_factory=list)
    special_requirements: List[str] = Field(
        default_factory=list,
        description="Applies to both venue and AV agents, e.g. 'livestream', 'wheelchair accessible'.",
    )
    existing_venue: Optional[ExistingVenue] = Field(
        default=None,
        description="If set, the orchestrator skips VenueAgent and uses this venue directly.",
    )

## 3. Trace logger

Records a timestamped event per agent handoff so the DAG's execution order - and the overlap of
parallel branches - can be inspected after the fact.

In [5]:
class TraceLogger:
    """Collects timestamped trace events so the DAG's execution - including the overlap of
    parallel branches - can be inspected after the fact."""

    def __init__(self):
        self.events: List[Dict[str, Any]] = []
        self._t0 = time.perf_counter()

    def log(self, actor: str, message: str) -> None:
        self.events.append(
            {"t": round(time.perf_counter() - self._t0, 3), "actor": actor, "message": message}
        )

    def render(self) -> str:
        return "\n".join(
            f"[t={e['t']:>6.3f}s] {e['actor']:<14} {e['message']}" for e in self.events
        )

## 4. The orchestrator

Routes an `EventRequest` through the DAG described below:

- **Branch 1 (sequential):** resolve the venue (`VenueAgent`, or skipped if `existing_venue` is
  set) → feed the chosen venue's area into `AVVendorCoordinationAgent`.
- **Branch 2 (independent):** `CateringLogisticsAgent` needs only the event's own
  type/headcount/budget/meal details, no venue info, so it runs **concurrently with Branch 1** on
  a second thread.
- **Join:** once both branches complete, `AgendaAssemblyAgent` combines venue + catering + AV into
  the final `RunOfShow`.

Budget is split across agents with a simple fixed-percentage heuristic (`VENUE_BUDGET_SHARE`,
etc.) - a real system might negotiate this dynamically, but a fixed split is enough to
demonstrate the routing / parallelism / conditional logic this notebook focuses on.

In [6]:
VENUE_BUDGET_SHARE = 0.45
CATERING_BUDGET_SHARE = 0.35
AV_BUDGET_SHARE = 0.20


class EventPlanningOrchestrator:
    """Central orchestrator: interprets one `EventRequest`, decides which specialist agents to
    call (and in what order / in parallel), routes each agent's output into the next agent's
    input, and assembles a final `RunOfShow`.
    """

    def __init__(
        self,
        venue_agent: VenueAgent,
        catering_agent: CateringLogisticsAgent,
        av_agent: AVVendorCoordinationAgent,
        agenda_agent: AgendaAssemblyAgent,
        trace: Optional[TraceLogger] = None,
    ):
        self._venue_agent = venue_agent
        self._catering_agent = catering_agent
        self._av_agent = av_agent
        self._agenda_agent = agenda_agent
        self.trace = trace or TraceLogger()

    def _resolve_venue(self, request: EventRequest) -> VenueRecommendation:
        if request.existing_venue is not None:
            self.trace.log(
                "orchestrator",
                "existing_venue provided -> skipping VenueAgent (conditional routing)",
            )
            ev = request.existing_venue
            return VenueRecommendation(
                recommended_venues=[
                    VenueOption(
                        name=ev.name,
                        area=ev.area,
                        capacity=ev.capacity,
                        estimated_cost_usd=0.0,
                        amenities=ev.amenities,
                        rationale="Already booked by the client - not sourced by VenueAgent.",
                    )
                ],
                assumptions=["Venue already secured by the client; VenueAgent was not invoked."],
            )

        self.trace.log("venue_agent", "invoked")
        result = self._venue_agent.run(
            VenueRequest(
                event_type=request.event_type,
                headcount=request.headcount,
                budget_usd=request.budget_usd * VENUE_BUDGET_SHARE,
                location=request.location,
                event_date=request.event_date,
                special_requirements=request.special_requirements,
            )
        )
        self.trace.log("venue_agent", "completed")
        return result

    def _run_catering(self, request: EventRequest) -> CateringPlan:
        self.trace.log("catering_agent", "invoked (parallel branch)")
        result = self._catering_agent.run(
            CateringRequest(
                event_type=request.event_type,
                headcount=request.headcount,
                budget_usd=request.budget_usd * CATERING_BUDGET_SHARE,
                meal_period=request.meal_period,
                dietary_restrictions=request.dietary_restrictions,
            )
        )
        self.trace.log("catering_agent", "completed (parallel branch)")
        return result

    def _run_venue_then_av(self, request: EventRequest) -> Tuple[VenueRecommendation, AVPlan]:
        venue = self._resolve_venue(request)
        selected = venue.recommended_venues[0]
        # VenueOption has no dedicated "venue category" field - `area` is the best available
        # proxy for AVRequest.venue_type (e.g. 'ballroom', 'rooftop').
        self.trace.log(
            "av_agent", f"invoked (routed venue choice '{selected.name}' / area '{selected.area}')"
        )
        av = self._av_agent.run(
            AVRequest(
                event_type=request.event_type,
                headcount=request.headcount,
                budget_usd=request.budget_usd * AV_BUDGET_SHARE,
                venue_type=selected.area,
                special_requirements=request.special_requirements,
            )
        )
        self.trace.log("av_agent", "completed")
        return venue, av

    def run(self, request: EventRequest) -> RunOfShow:
        self.trace.log("orchestrator", f"received request for '{request.event_name}'")

        with ThreadPoolExecutor(max_workers=2) as pool:
            venue_av_future = pool.submit(self._run_venue_then_av, request)
            catering_future = pool.submit(self._run_catering, request)
            venue, av = venue_av_future.result()
            catering = catering_future.result()

        self.trace.log("agenda_agent", "invoked (joins venue + catering + av)")
        run_of_show = self._agenda_agent.run(
            AgendaRequest(
                event_name=request.event_name,
                event_date=request.event_date,
                start_time=request.start_time,
                end_time=request.end_time,
                venue=venue,
                catering=catering,
                av=av,
            )
        )
        self.trace.log("agenda_agent", "completed")
        self.trace.log("orchestrator", "final event plan assembled")
        return run_of_show

## 5. Agent-flow diagram (DAG)

```
EventRequest
     │
     ├───────────────────────────────┬─────────────────────────────┐
     │                                │                             │
     ▼                                │                             ▼
┌──────────────────────────┐          │                 ┌───────────────────────────┐
│ existing_venue set?       │          │                 │  CateringLogisticsAgent   │
│  yes -> skip VenueAgent   │          │   (runs          │  (independent of venue)  │
│  no  -> VenueAgent        │          │   concurrently   └─────────────┬─────────────┘
└─────────────┬─────────────┘          │   with Branch 1)                │
              │ venue choice (area)    │                                 │
              ▼                        │                                 │
┌───────────────────────────┐          │                                 │
│ AVVendorCoordinationAgent  │◀─────────┘                                 │
└─────────────┬─────────────┘                                            │
              │ av plan                                                  │
              └───────────────────────────┬───────────────────────────────┘
                                          ▼
                              ┌───────────────────────────┐
                              │   AgendaAssemblyAgent      │   (join: venue + catering + av)
                              └─────────────┬─────────────┘
                                            ▼
                                   Final RunOfShow
```

```mermaid
flowchart TD
    REQ[EventRequest] --> COND{existing_venue set?}
    COND -- yes --> SKIP[Skip VenueAgent]
    COND -- no --> VENUE[VenueAgent]
    SKIP --> AV[AVVendorCoordinationAgent]
    VENUE --> AV
    REQ -. parallel branch .-> CATER[CateringLogisticsAgent]
    AV --> AGENDA[AgendaAssemblyAgent]
    CATER --> AGENDA
    AGENDA --> PLAN[Final RunOfShow]
```

## 6. Demo run 1 - full DAG, parallel venue/AV + catering branches

A 200-person product launch with no pre-booked venue, so every agent is invoked. A `FakeLLMClient`
with a small artificial `delay_seconds` simulates real latency, which is what makes the parallel
branch measurably faster than a naive sequential call order.

In [7]:
run1_backend = FakeLLMClient(
    canned_responses={
        "venue_recommendation_agent": {
            "recommended_venues": [
                {
                    "name": "Riverside Conference Center",
                    "area": "ballroom",
                    "capacity": 220,
                    "estimated_cost_usd": 48000,
                    "amenities": ["on-site parking", "AV rigging points"],
                    "rationale": "Ballroom space comfortably exceeds the 200-person headcount within budget.",
                }
            ],
            "assumptions": ["Rate assumes a weekday booking."],
        },
        "catering_logistics_agent": {
            "service_style": "plated",
            "menu_items": [
                {"course": "main", "description": "Herb-roasted chicken with seasonal vegetables", "dietary_tags": ["gluten-free"]},
                {"course": "main", "description": "Grilled portobello & quinoa stack", "dietary_tags": ["vegan", "gluten-free"]},
                {"course": "dessert", "description": "Seasonal fruit tart", "dietary_tags": ["vegan"]},
            ],
            "staffing_notes": "8 servers + 2 captains for a 200-guest plated lunch.",
            "estimated_cost_usd": 42000,
            "cost_per_head_usd": 210,
            "assumptions": [],
        },
        "av_vendor_coordination_agent": {
            "av_setup": [
                {"category": "audio", "description": "Line-array PA with 4 wireless handheld mics", "quantity": 4},
                {"category": "video", "description": "2 confidence monitors + 1 main projection screen", "quantity": 3},
                {"category": "connectivity", "description": "Dedicated livestream encoder + wifi hotspot", "quantity": 1},
            ],
            "recommended_vendors": [
                {"vendor_role": "AV production company", "notes": "Full-service crew for setup, show-call, and teardown."},
                {"vendor_role": "livestream provider", "notes": "Handles the multi-camera livestream feed for remote attendees."},
            ],
            "estimated_cost_usd": 24000,
            "setup_lead_time_hours": 6,
            "assumptions": [],
        },
        "agenda_assembly_agent": {
            "event_name": "Nova Product Launch",
            "event_date": "2026-11-18",
            "selected_venue": "Riverside Conference Center",
            "run_of_show": [
                {"start_time": "07:00", "end_time": "09:00", "activity": "AV setup & sound check", "owner": "AV crew", "location": "Ballroom", "notes": None},
                {"start_time": "09:00", "end_time": "09:30", "activity": "Registration & welcome coffee", "owner": "Host", "location": "Foyer", "notes": None},
                {"start_time": "09:30", "end_time": "11:30", "activity": "Keynote & product reveal", "owner": "Host", "location": "Ballroom", "notes": "Livestreamed"},
                {"start_time": "11:30", "end_time": "13:00", "activity": "Plated lunch service", "owner": "Catering team", "location": "Ballroom", "notes": None},
                {"start_time": "13:00", "end_time": "14:00", "activity": "Teardown", "owner": "AV crew", "location": "Ballroom", "notes": None},
            ],
            "overall_notes": ["Vegan and gluten-free plated options confirmed with catering."],
        },
    },
    delay_seconds=0.3,
)

orchestrator1 = EventPlanningOrchestrator(
    venue_agent=VenueAgent(run1_backend),
    catering_agent=CateringLogisticsAgent(run1_backend),
    av_agent=AVVendorCoordinationAgent(run1_backend),
    agenda_agent=AgendaAssemblyAgent(run1_backend),
)

run1_request = EventRequest(
    event_name="Nova Product Launch",
    event_type=EventType.PRODUCT_LAUNCH,
    headcount=200,
    budget_usd=250000,
    location="Austin",
    event_date="2026-11-18",
    start_time="09:00",
    end_time="14:00",
    meal_period="lunch",
    dietary_restrictions=["vegan", "gluten-free"],
    special_requirements=["livestream"],
)

start = time.perf_counter()
run1_plan = orchestrator1.run(run1_request)
elapsed1 = time.perf_counter() - start

sequential_estimate1 = 4 * run1_backend.delay_seconds  # venue + catering + av + agenda, back-to-back
print(f"Wall-clock time: {elapsed1:.2f}s  (naive sequential estimate: {sequential_estimate1:.2f}s)\n")
print("=== Trace log ===")
print(orchestrator1.trace.render())
print("\n=== Final event plan ===")
print(run1_plan.model_dump_json(indent=2))

assert elapsed1 < sequential_estimate1, "parallel branch should beat the naive sequential estimate"

Wall-clock time: 0.92s  (naive sequential estimate: 1.20s)

=== Trace log ===
[t= 0.000s] orchestrator   received request for 'Nova Product Launch'
[t= 0.001s] venue_agent    invoked
[t= 0.002s] catering_agent invoked (parallel branch)
[t= 0.309s] catering_agent completed (parallel branch)
[t= 0.311s] venue_agent    completed
[t= 0.311s] av_agent       invoked (routed venue choice 'Riverside Conference Center' / area 'ballroom')
[t= 0.613s] av_agent       completed
[t= 0.614s] agenda_agent   invoked (joins venue + catering + av)
[t= 0.916s] agenda_agent   completed
[t= 0.916s] orchestrator   final event plan assembled

=== Final event plan ===
{
  "event_name": "Nova Product Launch",
  "event_date": "2026-11-18",
  "selected_venue": "Riverside Conference Center",
  "run_of_show": [
    {
      "start_time": "07:00",
      "end_time": "09:00",
      "activity": "AV setup & sound check",
      "owner": "AV crew",
      "location": "Ballroom",
      "notes": null
    },
    {
      "start

## 7. Demo run 2 - conditional routing: client already has a booked venue

A smaller corporate offsite where the client already secured a venue. `existing_venue` is set, so
the orchestrator must **skip `VenueAgent`** entirely - proven below both by the trace log and by
the fact that `run2_backend` has no canned response registered for `venue_recommendation_agent`
at all (a call would raise `KeyError` if the conditional branch were broken).

In [8]:
run2_backend = FakeLLMClient(
    canned_responses={
        "catering_logistics_agent": {
            "service_style": "buffet",
            "menu_items": [
                {"course": "main", "description": "Seasonal grain bowls with roasted vegetables", "dietary_tags": ["vegetarian"]},
                {"course": "main", "description": "Herb-marinated grilled chicken", "dietary_tags": []},
                {"course": "dessert", "description": "Assorted mini desserts", "dietary_tags": ["vegetarian"]},
            ],
            "staffing_notes": "3 buffet attendants for a 60-guest lunch.",
            "estimated_cost_usd": 10500,
            "cost_per_head_usd": 175,
            "assumptions": [],
        },
        "av_vendor_coordination_agent": {
            "av_setup": [
                {"category": "audio", "description": "Portable PA with 2 wireless lapel mics", "quantity": 2},
                {"category": "video", "description": "55-inch display for slides", "quantity": 1},
            ],
            "recommended_vendors": [
                {"vendor_role": "AV production company", "notes": "Small-footprint crew; venue already has an in-house AV rack."},
            ],
            "estimated_cost_usd": 3500,
            "setup_lead_time_hours": 2,
            "assumptions": ["Reuses the client's existing rooftop AV rack where possible."],
        },
        "agenda_assembly_agent": {
            "event_name": "Quarterly Offsite",
            "event_date": "2026-12-02",
            "selected_venue": "Client HQ - Rooftop Hall",
            "run_of_show": [
                {"start_time": "08:30", "end_time": "10:00", "activity": "AV setup", "owner": "AV crew", "location": "Rooftop Hall", "notes": None},
                {"start_time": "10:00", "end_time": "12:00", "activity": "Strategy session", "owner": "Host", "location": "Rooftop Hall", "notes": None},
                {"start_time": "12:00", "end_time": "13:00", "activity": "Buffet lunch", "owner": "Catering team", "location": "Rooftop Hall", "notes": None},
                {"start_time": "13:00", "end_time": "16:00", "activity": "Breakout workshops", "owner": "Host", "location": "Rooftop Hall", "notes": None},
            ],
            "overall_notes": ["Vegetarian buffet options confirmed."],
        },
    },
    delay_seconds=0.2,
)
# Note: no "venue_recommendation_agent" entry here - VenueAgent must never be called in this run.

orchestrator2 = EventPlanningOrchestrator(
    venue_agent=VenueAgent(run2_backend),
    catering_agent=CateringLogisticsAgent(run2_backend),
    av_agent=AVVendorCoordinationAgent(run2_backend),
    agenda_agent=AgendaAssemblyAgent(run2_backend),
)

run2_request = EventRequest(
    event_name="Quarterly Offsite",
    event_type=EventType.CORPORATE_OFFSITE,
    headcount=60,
    budget_usd=90000,
    location="Denver",
    event_date="2026-12-02",
    start_time="10:00",
    end_time="16:00",
    meal_period="lunch",
    dietary_restrictions=["vegetarian"],
    existing_venue=ExistingVenue(
        name="Client HQ - Rooftop Hall",
        area="rooftop",
        capacity=80,
        amenities=["in-house AV rack", "skyline view"],
    ),
)

run2_plan = orchestrator2.run(run2_request)
print("=== Trace log ===")
print(orchestrator2.trace.render())
print("\n=== Final event plan ===")
print(run2_plan.model_dump_json(indent=2))

assert "venue_recommendation_agent" not in {c["schema_name"] for c in run2_backend.calls}, (
    "VenueAgent must be skipped"
)
assert any("skipping VenueAgent" in e["message"] for e in orchestrator2.trace.events), (
    "trace should record the conditional skip"
)
assert run2_plan.selected_venue == "Client HQ - Rooftop Hall"
print("\nConditional-routing checks passed.")

=== Trace log ===
[t= 0.000s] orchestrator   received request for 'Quarterly Offsite'
[t= 0.001s] orchestrator   existing_venue provided -> skipping VenueAgent (conditional routing)
[t= 0.001s] av_agent       invoked (routed venue choice 'Client HQ - Rooftop Hall' / area 'rooftop')
[t= 0.002s] catering_agent invoked (parallel branch)
[t= 0.202s] av_agent       completed
[t= 0.203s] catering_agent completed (parallel branch)
[t= 0.203s] agenda_agent   invoked (joins venue + catering + av)
[t= 0.404s] agenda_agent   completed
[t= 0.404s] orchestrator   final event plan assembled

=== Final event plan ===
{
  "event_name": "Quarterly Offsite",
  "event_date": "2026-12-02",
  "selected_venue": "Client HQ - Rooftop Hall",
  "run_of_show": [
    {
      "start_time": "08:30",
      "end_time": "10:00",
      "activity": "AV setup",
      "owner": "AV crew",
      "location": "Rooftop Hall",
      "notes": null
    },
    {
      "start_time": "10:00",
      "end_time": "12:00",
      "activi

## 8. Checks - proving the parallel branch actually overlaps

Beyond the wall-clock comparison in run 1, the trace log's timestamps let us prove the catering
branch *started* before the venue branch *finished* - direct evidence the two branches ran
concurrently rather than back-to-back.

In [9]:
venue_completed_t = next(
    e["t"] for e in orchestrator1.trace.events if e["actor"] == "venue_agent" and e["message"] == "completed"
)
catering_invoked_t = next(
    e["t"] for e in orchestrator1.trace.events if e["actor"] == "catering_agent" and "invoked" in e["message"]
)

assert catering_invoked_t < venue_completed_t, (
    "catering branch should start before the venue branch finishes (proof of overlap)"
)
print(
    "Parallelism overlap check passed: "
    f"catering started at t={catering_invoked_t}s, venue finished at t={venue_completed_t}s"
)

Parallelism overlap check passed: catering started at t=0.002s, venue finished at t=0.311s


## Summary

| Deliverable | Where |
|---|---|
| Orchestrator routing between all four agents | `EventPlanningOrchestrator` (Section 4) |
| Agent-flow diagram (DAG) | Section 5 (ASCII + Mermaid) |
| End-to-end trace log for a sample request | Run 1's `orchestrator1.trace.render()` output (Section 6) |
| Parallel execution path | `ThreadPoolExecutor` fork of the venue/AV branch vs. the catering branch in `EventPlanningOrchestrator.run`, proven by the Section 8 timestamp check |
| Conditional routing decision | `EventRequest.existing_venue` short-circuits `VenueAgent` in `_resolve_venue`, exercised in Run 2 (Section 7) |